# Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path
from hydra.utils import instantiate
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils.notebook_setup import init_nlp_notebook # noqa: E402
cfg = init_nlp_notebook()

# Chunking Audit

In [ ]:
from llama_index.core import Document
from src.jobs.rag_update_job import clean_text

# Берем тестовый сырой текст (или загружаем из cfg.paths.data_dir)
raw_text = """
Анализ данных — это процесс инспекции, очистки и моделирования данных. 
<br>Цель состоит в обнаружении полезной информации! 
Подробнее: https://example.com/data
"""

# Имитируем логику из rag_update_job.py
cleaned_text = clean_text(raw_text)
test_doc = Document(text=cleaned_text)

# Инициализируем индексатор для доступа к сплиттеру (без сохранения)
indexer = instantiate(cfg.rag.indexer)
nodes = indexer.Settings.node_parser.get_nodes_from_documents([test_doc])

print(f"Original Cleaned Text:\n{cleaned_text}\n")
print("-" * 40)
for i, node in enumerate(nodes):
    print(f"Chunk {i+1} (Length: {len(node.text)}):\n{node.text}\n")

# Indexing

In [ ]:
# Если базы еще нет, создаем её прямо здесь
# indexer.build_and_save_index() будет читать данные из documents_dir
try:
    retriever = instantiate(cfg.rag.retriever)
    print("Векторная БД успешно загружена.")
except Exception as e:
    print(f"БД не найдена. Собираем индекс...\n{e}")
    indexer = instantiate(cfg.rag.indexer)
    indexer.build_and_save_index() # Загрузит сырые файлы и сохранит в FAISS
    retriever = instantiate(cfg.rag.retriever)

# Sanity Check Retrieval

In [ ]:
test_query = "Что такое RAG и как он работает?"

# Используем твой базовый метод
context = retriever.retrieve_context(test_query)

print(f"QUERY: {test_query}\n")
print(f"RETRIEVED CONTEXT (Top-{retriever.similarity_top_k}):\n")
print(context)

# Score Distribution


In [ ]:
import numpy as np
import seaborn as sns

queries = [
    "Как настроить параметры Llama-index?",
    "Какие существуют методы классификации?",
    "Привет, как дела?" # Запрос "вне домена", чтобы увидеть плохие скоры
]

all_scores = []

for q in queries:
    # Обращаемся напрямую к внутреннему объекту LlamaIndex
    nodes_with_score = retriever.retriever.retrieve(q)
    scores = [node.score for node in nodes_with_score]
    all_scores.extend(scores)
    
    print(f"Query: '{q}'")
    for i, node in enumerate(nodes_with_score):
        print(f"  [{i+1}] Score: {node.score:.4f} | Text: {node.get_content()[:60]}...")
    print("-" * 50)

# Визуализируем распределение скоров
plt.figure(figsize=(8, 4))
sns.histplot(all_scores, bins=15, kde=True)
plt.title("Distribution of Retrieval Scores (Distances)")
plt.xlabel("Score")
plt.ylabel("Count")
plt.show()

# На основе этого графика ты сможешь решить:
# "Если Score < 0.3 (или > 1.5 для L2), значит контекст нерелевантен, и LLM должна отвечать 'Я не знаю'".